## XBRL US API - FERC schedule by all filers

### Authenticate for access token 
Click in the gray code cell below, then click the Run button above to execute the cell. Type your XBRL US Web account email, account password, Client ID, and secret as noted, pressing the Enter key on the keyboard after each entry.

XBRL US limits records returned for a query to improve efficiency; this script loops to collect all data from the Public Filings Database for a query. **Non-members might not be able to return all data for a query** - join XBRL US for comprehensive access - https://xbrl.us/join.

In [ ]:
# @title Run cell to authenticate XBRL API {\"display-mode\":\"form\"}\n",
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode


class tokenInfoClass:
    access_token = None
    refresh_token = None
    email = None
    username = None
    client_id = None
    client_secret = None
    url = 'https://api.xbrl.us/oauth2/token'
    headers = {"Content-Type": "application/x-www-form-urlencoded"}

def refresh(info):
    refresh_auth = {
                'client_id': info.client_id,
                'client_secret' : info.client_secret,
                'grant_type' : 'refresh_token',
                'platform' : 'ipynb',
                'refresh_token' : info.refresh_token
                }
    refreshres = requests.post(info.url, data=refresh_auth, headers=info.headers)
    refresh_json = refreshres.json()
    info.access_token = refresh_json.get('access_token')
    info.refresh_token = refresh_json.get('refresh_token')
    print('Your access token(%s) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.' % (info.access_token))
    return info

tokenInfo = tokenInfoClass()

# Helper to prompt only if value is missing
def prompt_if_missing(value, prompt_text, secret=False):
    if value:
        return value
    if secret:
        return getpass.getpass(prompt=prompt_text)
    return input(prompt_text)

# Load credentials (if .json exists)
creds = {}
if os.path.exists('creds.json'):
    try:
        with open('creds.json', 'r') as f:
            creds = json.load(f)
        print("Loaded .json")
    except Exception as e:
        print("Warning: failed to read from .json:", e)

if creds:
    # Try nested prod object first
    selected = None
    if isinstance(creds.get('prod'), dict):
        selected = creds['prod']
    
    # Next, try prod-prefixed keys
    if not selected:
        selected = {}
        keys = ['email', 'password', 'client_id', 'client_secret']
        for k in keys:
            prefixed_key = 'prod' + k
            if creds.get(prefixed_key):
                selected[k] = creds.get(prefixed_key)
            # fall back to top-level key if prod variant not found
            elif creds.get(k):
                selected[k] = creds.get(k)
    
    # Verify we have all required keys
    if not all(selected.get(k) for k in ('email', 'password', 'client_id', 'client_secret')):
        # Fill in missing values from prompts
        selected = {
            'email': selected.get('email'),
            'password': selected.get('password'),
            'client_id': selected.get('client_id'),
            'client_secret': selected.get('client_secret')
        }
    
    # Assign values, prompting for any missing ones
    tokenInfo.email = prompt_if_missing(selected.get('email'), 'Enter your XBRL US Web account email: ')
    tokenInfo.password = prompt_if_missing(selected.get('password'), 'Password: ', secret=True)
    tokenInfo.client_id = prompt_if_missing(selected.get('client_id'), 'Client ID: ', secret=True)
    tokenInfo.client_secret = prompt_if_missing(selected.get('client_secret'), 'Secret: ', secret=True)

    print('Using credentials from .json as available.')
else:
    # No creds.json — prompt the user
    tokenInfo.email = input('Enter your XBRL US Web account email: ')
    tokenInfo.password = getpass.getpass(prompt='Password: ')
    tokenInfo.client_id = getpass.getpass(prompt='Client ID: ')
    tokenInfo.client_secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : tokenInfo.email,
            'client_id': tokenInfo.client_id,
            'client_secret' : tokenInfo.client_secret,
            'password' : tokenInfo.password,
            'grant_type' : 'password',
            'platform' : 'ipynb' }

#print(body_auth)

payload = urlencode(body_auth)
res = requests.request("POST", tokenInfo.url, data=payload, headers=tokenInfo.headers)
auth_json = res.json()

if 'error' in auth_json:
    print("\n\nThere was a problem generating the access token: %s  Run the first cell again and enter the credentials." % (auth_json.get('error_description', auth_json)))
else:
    tokenInfo.access_token = auth_json.get('access_token')
    tokenInfo.refresh_token = auth_json.get('refresh_token')
    if tokenInfo.access_token and tokenInfo.refresh_token:
        print ("\n\nYour access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. \n\nFor now, skip ahead to the section 'Make a Query'.".format(tokenInfo.email))
    else:
        print("\n\nAuthentication completed but tokens were not returned. Response: {}".format(auth_json))

#print(vars(tokenInfo))
if tokenInfo.access_token and tokenInfo.refresh_token:
    print('\n\naccess token: ' + tokenInfo.access_token + ' refresh token: ' + tokenInfo.refresh_token + ' for ' + tokenInfo.email)
else:
    print('\n\nNo access token was generated. Check the messages above for errors.')

### Make a query 
After the access token confirmation appears above, you can modify the query below and use the **_Cell >> Run_** menu option with the cell **immediately below this text** to run the query for updated results. 

The sample results are 2 years of data for all entity codes filing data on Report 1 for _232 - Schedule - Other Regulatory Assets (Account 182.3)_, and may take several minutes to recreate.  To test for results quickly, uncomment the **_entity.code_** line in the params section to shorten the list. Open the commented query URL for unique Statements in a browser to get a list of schedules in the report, then update the corresponding variable in the query.
  
Refer to XBRL API documentation at https://xbrlus.github.io/xbrl-api/#/Facts/getFactDetails for other endpoints and parameters to filter and return. 

In [5]:
## Example Queries (these can open in a browser for quick reference)
# query taxonomies published for a specific year
# https://api.xbrl.us/api/v1/dts/search?fields=dts.id,dts.version.sort(DESC),dts.taxonomy-name.sort(DESC)&dts.version=2023

# query unique Statements in the 2023 FERC Form 1 Taxonomy
# https://api.xbrl.us/api/v1/dts/900342/network/search?network.link-name=presentationLink&fields=network.role-description.sort(ASC),dts.id&unique

# query Form 1 filer entity codes:
# https://api.xbrl.us/api/v1/report/search?report.document-type=1&fields=report.entity-name.sort(ASC),entity.code&unique


### Define the parameters for the filter and fields to be returned

# Define endpoint (common values: fact, entity, report, cube, label, concept, relationship - see https://xbrlus/github.io/xbrl-api for additional endpoint options)

endpoint = 'fact'

document_type = [
    '1'
    ]

entity_codes = [
    'C003184',
    'C000041',
    'C009068'
    ]

years = [
    '2023',
    '2024',
    '2025'
    ]

elements = [
    'NetIncomeLoss',
    'ProprietaryCapital'
]

# Define data fields to return (multi-sort based on order)

fields = [ # this is the list of the characteristics of the data being returned by the query
    'entity.name.sort(ASC)',
    'entity.code',
    'period.fiscal-year.sort(DESC)',
    'period.fiscal-period',
    'concept.local-name.sort(ASC)',
    'fact.value',
    'unit',
    'dimensions.count'
        ]

# Set unique rows as True of False (True drops any duplicate rows)
unique = True

# Limit the number of rows displayed by the notebook (does not impact the data frame)
rows_to_display = 20 # Set as '' to display all rows in the notebook

# Below is the list of what's being queried using the search endpoint.
 
params = { 
         'report.document-type': ','.join(document_type),
         #'entity.code': ','.join(entity_codes),
         'period.fiscal-year': ','.join(years),
         'period.fiscal-period': 'Y',
         'concept.local-name': ','.join(elements),
         'fact.has-dimensions': 'FALSE',
         'fact.ultimus': 'TRUE',
         'fields': ','.join(fields)
         }

search_endpoint = 'https://api.xbrl.us/api/v1/' + endpoint + '/search'
if unique:
    search_endpoint += "?unique"
orig_fields = params['fields']

query_url = search_endpoint + ('&' if unique else '?') + urlencode(params)

print('Run the next cell to query the database for results for the query below. If you want to change the query, edit the parameters above and rerun this cell.\n\nQuery URL: ' + query_url)

Run the next cell to query the database for results for the query below. If you want to change the query, edit the parameters above and rerun this cell.

Query URL: https://api.xbrl.us/api/v1/fact/search?unique&report.document-type=1&period.fiscal-year=2023%2C2024%2C2025&period.fiscal-period=Y&concept.local-name=NetIncomeLoss%2CProprietaryCapital&fact.has-dimensions=FALSE&fact.ultimus=TRUE&fields=entity.name.sort%28ASC%29%2Centity.code%2Cperiod.fiscal-year.sort%28DESC%29%2Cperiod.fiscal-period%2Cconcept.local-name.sort%28ASC%29%2Cfact.value%2Cunit%2Cdimensions.count


In [6]:
# @title Run loop for results {"display-mode":"form"}

### Execute the query with loop for all results 
### THIS SECTION DOES NOT NEED TO BE EDITED

import time

offset_value = 0
res_df = []
count = 0
query_start = datetime.now()
printed = False
run_query = True

while True:
    if not printed:
        print("On", query_start.strftime("%c"), tokenInfo.email, "(client ID:", str(tokenInfo.client_id.split('-')[0]), "...) started the query and")
        printed = True
    retry = 0
    while retry < 3:
        res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(tokenInfo.access_token)})
        try:
            res_json = res.json()
        except ValueError:
            print("The server returned an empty or invalid response (HTTP {}), likely because the query is taking too long. Retrying...".format(res.status_code))
            res_json = {'error': 'invalid_response', 'error_description': 'Non-JSON response from server'}
        if 'error' in res_json:
            if res_json['error_description'] == 'Bad or expired token':
                tokenInfo = refresh(tokenInfo)
            elif res_json['error_description'] == 'Non-JSON response from server':
                time.sleep(5)
            else: 
                print('There was an error: {}'.format(res_json['error_description']))
                run_query = False
                break
        else: 
		        break
        retry +=1
        if retry >= 3:
            print("Can't get a valid response from the server after 3 attempts. The query may be too large or timing out - try narrowing it (e.g., set entity.code or shorten the years list) and rerun.")
            run_query = False

    if not run_query:
       break

    print("up to", str(offset_value + res_json['paging']['limit']), "records are found so far ...")

    res_df += res_json['data']

    if res_json['paging']['count'] < res_json['paging']['limit']:
        print(" - this set contained fewer than the", res_json['paging']['limit'], "possible, only", str(res_json['paging']['count']), "records.")
        break
    else: 
        offset_value += res_json['paging']['limit'] 
        if 100 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 10 * res_json['paging']['limit']:
                        break 
        elif 500 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 4 * res_json['paging']['limit']:
                        break 
        params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    total_rows = len(index)
    your_limit = res_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"
    
    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    
    print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with  ", str(total_rows), "  rows returned in " + str(time_taken) + " for \n" +  urllib.parse.unquote(res.url))
    
    df = pd.DataFrame(res_df)
    # the format truncates the HTML display of numerical values to two decimals; .csv data is unaffected
    pd.options.display.float_format = '{:,.2f}'.format
    display(HTML(df.to_html(max_rows=rows_to_display)))

On Mon Oct 12 20:57:00 2026 david.tauriello@xbrl.us (client ID: 33cccd22 ...) started the query and
up to 5000 records are found so far ...
 - this set contained fewer than the 5000 possible, only 1106 records.

At Mon Oct 12 20:59:22 2026, the query finished with   1106   rows returned in 0:02:21.100973 for 
https://api.xbrl.us/api/v1/fact/search?unique&report.document-type=1&period.fiscal-year=2023,2024,2025&period.fiscal-period=Y&concept.local-name=NetIncomeLoss,ProprietaryCapital&fact.has-dimensions=FALSE&fact.ultimus=TRUE&fields=entity.name.sort(ASC),entity.code,period.fiscal-year.sort(DESC),period.fiscal-period,concept.local-name.sort(ASC),fact.value,unit,dimensions.count


,entity.name,entity.code,period.fiscal-year,period.fiscal-period,concept.local-name,fact.value,unit,dimensions.count
0,"AEP Appalachian Transmission Company, Inc.",C001702,2025,Y,NetIncomeLoss,6774271,USD,0
1,"AEP Appalachian Transmission Company, Inc.",C001702,2024,Y,NetIncomeLoss,5139353,USD,0
2,"AEP Appalachian Transmission Company, Inc.",C001702,2024,Y,ProprietaryCapital,52588035,USD,0
3,"AEP Appalachian Transmission Company, Inc.",C001702,2023,Y,NetIncomeLoss,4725339,USD,0
4,"AEP Appalachian Transmission Company, Inc.",C001702,2023,Y,ProprietaryCapital,47316052,USD,0
5,AEP Generating Company,C003184,2025,Y,NetIncomeLoss,8475322,USD,0
6,AEP Generating Company,C003184,2024,Y,NetIncomeLoss,11330381,USD,0
7,AEP Generating Company,C003184,2024,Y,ProprietaryCapital,140882519,USD,0
8,AEP Generating Company,C003184,2023,Y,NetIncomeLoss,10813682,USD,0
9,AEP Generating Company,C003184,2023,Y,ProprietaryCapital,147110398,USD,0


In [ ]:
# If you run this program locally, you can save the output to a file on your computer (modify D:\results.csv to your system)
df.to_csv(r"D:\results.csv",sep=",")

# Google Colab users - comment out the line above and uncomment the code below to save the data frame as a .csv in your Google Drive

#from google.colab import drive
#drive.mount('drive')
#df.to_csv('data.csv')
#!cp data.csv "drive/My Drive/"